# raw_data_transform_v2
**Purpose:** Re-label the RockYou dataset using the `password-strength` library instead of zxcvbn.

The original pipeline used zxcvbn, whose scoring is heavily length-biased. This notebook replaces that labeling step with `PasswordStats.strength()`, which is entropy-density based — a short password with high character variety can outscore a long repetitive one.

All other steps (cleaning, balancing, saving) are identical to the original `raw_data_transform.ipynb`.

Output: `rockyou_balanced_pwstrength.csv` — same structure as `rockyou_balanced.csv`, drop-in compatible with `final_notebook_v2.ipynb`.

In [1]:
# Cell 1 — Install password-strength library if not already installed
# Only needs to run once. Safe to skip on subsequent runs.
# zxcvbn_rs_py is NOT imported here — this notebook intentionally replaces it.

import subprocess
subprocess.run(['pip', 'install', 'password-strength', 'pandarallel'], check=True)

CompletedProcess(args=['pip', 'install', 'password-strength', 'pandarallel'], returncode=0)

In [2]:
# Cell 2 — Imports

import pandas as pd
import numpy as np
import re
from password_strength import PasswordStats
from pandarallel import pandarallel

In [3]:
# Cell 3 — Load raw RockYou dataset
# Update the path below to match your local file location.
# latin-1 encoding is required — rockyou.txt contains non-UTF-8 bytes.
# on_bad_lines='skip' drops corrupted lines without crashing.

RAW_PATH = r'E:\major_project\datasets\rockyou.txt'

df = pd.read_table(
    RAW_PATH,
    names=['password'],
    encoding='latin-1',
    header=None,
    dtype={'password': str},
    on_bad_lines='skip'
)

print(f'Loaded {len(df):,} raw passwords')

Loaded 14,344,097 raw passwords


In [4]:
df.head()

,password
0,123456
1,12345
2,123456789
3,password
4,iloveyou


In [ ]:
# Cell 4 — Clean passwords
# Identical to original raw_data_transform.ipynb.
# Removes non-printable control characters, strips whitespace,
# and drops empty strings and nulls.

def clean_password(p):
    if not isinstance(p, str):
        return None
    # Remove non-printable characters (ASCII control codes)
    p = re.sub(r'[\x00-\x1F\x7F-\x9F]', '', p)
    # Strip leading and trailing whitespace
    p = p.strip()
    # Return None for empty strings so dropna() removes them
    return p if len(p) > 0 else None

df['password'] = df['password'].apply(clean_password)
df = df.dropna(subset=['password'])

# Remove passwords with invisible control characters that slipped through
df = df[~df['password'].str.contains(r'[\x00-\x1F\x7F-\x9F]', na=True, regex=True)]

# Remove duplicates
df.drop_duplicates(subset=['password'], inplace=True)

print(f'After cleaning: {len(df):,} passwords')

In [ ]:
# Cell 5 — Define the new labeling function using password-strength
#
# PasswordStats.strength() returns a float between 0.0 and 1.0.
# It is entropy-density based, not length-based:
#   - A short password with high character variety can score well
#   - A long repetitive password (e.g. 'aaaaaaaaaa') scores poorly
#   - weakness_factor penalises repeated patterns within the string
#
# Thresholds chosen to produce 3 classes:
#   0.00 - 0.33  ->  0 (Weak)
#   0.33 - 0.66  ->  1 (Medium)
#   0.66 - 1.00  ->  2 (Strong)
#
# These thresholds can be adjusted. Tighter thresholds (e.g. 0.4/0.7)
# will produce fewer Strong labels and more Weak ones, which may better
# reflect real-world password quality distributions.

WEAK_THRESHOLD   = 0.33
STRONG_THRESHOLD = 0.66

def get_strength_pwstats(pwd):
    try:
        score = PasswordStats(str(pwd)).strength()
        if score < WEAK_THRESHOLD:
            return 0  # Weak
        elif score < STRONG_THRESHOLD:
            return 1  # Medium
        else:
            return 2  # Strong
    except Exception:
        # If the library fails on an unusual character, default to Weak
        return 0

# Quick sanity check before running on the full dataset
test_cases = [
    ('password',         0),  # common word, should be Weak
    ('aaaaaaaaaa',       0),  # all same char, should be Weak
    ('Hello123',         1),  # mixed, should be Medium
    ('P@ssw0rd!9xQ#',    2),  # complex, should be Strong
]

print('Sanity check:')
all_pass = True
for pwd, expected in test_cases:
    result = get_strength_pwstats(pwd)
    status = 'PASS' if result == expected else 'FAIL'
    if status == 'FAIL':
        all_pass = False
    raw_score = round(PasswordStats(pwd).strength(), 3)
    print(f'  [{status}] "{pwd}" -> class {result} (raw score: {raw_score}, expected class {expected})')

if not all_pass:
    print('\nSome sanity checks failed. Consider adjusting WEAK_THRESHOLD / STRONG_THRESHOLD above.')
else:
    print('\nAll sanity checks passed. Safe to proceed.')

In [ ]:
# Cell 6 — Apply labeling to the full dataset using parallel processing
#
# pandarallel splits the work across CPU cores.
# On an i7 11th gen this should take roughly 10-15 minutes for 1.4M rows.
# If pandarallel causes issues, replace parallel_apply with regular apply
# (will be slower but identical output).

pandarallel.initialize(progress_bar=True)

print('Labeling passwords with password-strength library...')
print(f'Total passwords to label: {len(df):,}')

df['strength'] = df['password'].parallel_apply(get_strength_pwstats)

print('Labeling complete.')
print()
print('Raw class distribution:')
counts = df['strength'].value_counts().sort_index()
labels = {0: 'Weak', 1: 'Medium', 2: 'Strong'}
for cls, count in counts.items():
    pct = count / len(df) * 100
    print(f'  Class {cls} ({labels[cls]}): {count:,} ({pct:.1f}%)')

In [ ]:
# Cell 7 — Inspect score distribution before committing to thresholds
#
# This cell samples 10,000 passwords and plots the raw strength score
# distribution. Use this to verify the thresholds make sense for your data
# before running the full balance step.
# If the distribution is heavily skewed, adjust WEAK_THRESHOLD and
# STRONG_THRESHOLD in Cell 5 and re-run from Cell 6.

import matplotlib.pyplot as plt

sample = df['password'].sample(10000, random_state=42)
sample_scores = sample.apply(lambda p: PasswordStats(str(p)).strength())

plt.figure(figsize=(10, 4))
plt.hist(sample_scores, bins=50, color='steelblue', edgecolor='white')
plt.axvline(WEAK_THRESHOLD,   color='orange', linestyle='--', label=f'Weak threshold ({WEAK_THRESHOLD})')
plt.axvline(STRONG_THRESHOLD, color='green',  linestyle='--', label=f'Strong threshold ({STRONG_THRESHOLD})')
plt.title('password-strength score distribution (10k sample)')
plt.xlabel('strength() score')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Score percentiles (sample of 10k):')
for p in [10, 25, 50, 75, 90]:
    print(f'  {p}th percentile: {sample_scores.quantile(p/100):.3f}')

In [ ]:
# Cell 8 — Stratified sampling to balance classes
#
# Identical logic to original raw_data_transform.ipynb.
# Target 100k per class. If a class has fewer than 100k, take all of them.
#
# NOTE: Unlike the original which had 5 zxcvbn classes (0-4) before merging,
# we now have 3 classes directly. This avoids the original imbalance caused
# by merging {0,1}->Weak and {2,3}->Medium after sampling, which produced
# ~200k Weak, ~200k Medium, and ~100k Strong without correction.

TARGET_PER_CLASS = 100000

balanced_samples = []

print('Balancing classes...')
for cls in [0, 1, 2]:
    class_group = df[df['strength'] == cls]
    count = len(class_group)
    label = labels[cls]
    print(f'  Class {cls} ({label}): {count:,} available')

    if count >= TARGET_PER_CLASS:
        balanced_samples.append(class_group.sample(TARGET_PER_CLASS, random_state=42))
        print(f'    -> Sampled {TARGET_PER_CLASS:,}')
    else:
        # Rare class — take everything available
        balanced_samples.append(class_group)
        print(f'    -> Rare class, taking all {count:,}')

df_balanced = pd.concat(balanced_samples).reset_index(drop=True)

print()
print(f'Final balanced dataset: {len(df_balanced):,} rows')
print()
print('Final class distribution:')
final_counts = df_balanced['strength'].value_counts().sort_index()
for cls, count in final_counts.items():
    pct = count / len(df_balanced) * 100
    print(f'  Class {cls} ({labels[cls]}): {count:,} ({pct:.1f}%)')

In [ ]:
# Cell 9 — Final checks before saving
#
# Verify there are no nulls, the class column is correct type,
# and the structure matches what final_notebook_v2 expects.

assert df_balanced['password'].isnull().sum() == 0, 'Null passwords found'
assert df_balanced['strength'].isnull().sum() == 0, 'Null strength labels found'
assert set(df_balanced['strength'].unique()).issubset({0, 1, 2}), 'Unexpected class values'
assert len(df_balanced) > 0, 'Empty dataset'

print('All checks passed.')
print()
print('Dataset preview:')
df_balanced.sample(10, random_state=42)[['password', 'strength']]

In [ ]:
# Cell 10 — Save balanced dataset
#
# Saved to a new file — original rockyou_balanced.csv is NOT overwritten.
# Update the output path to match your directory structure.

OUTPUT_PATH = r'E:\major_project\datasets\rockyou_balanced_pwstrength.csv'

df_balanced.to_csv(OUTPUT_PATH, index=False)

print(f'Saved to: {OUTPUT_PATH}')
print(f'Rows: {len(df_balanced):,}')
print(f'Columns: {list(df_balanced.columns)}')
print()
print('Next step: open final_notebook_v2.ipynb and update the dataset path to this file.')